In [1]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch==0.9.4",
#     "torch==2.13.0",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-21 14:36 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [2]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "dataLoader_1784578578190_speech_commands_mfcc20_test.pt",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

Config loaded: {'dataset': 'dataLoader_1784578578190_speech_commands_mfcc20_test.pt', 'framework': 'snntorch_sim'}


In [3]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict,
    phase: str = "train",
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
                "phase": phase,
            }
        ),
        flush=True,
    )


In [4]:
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _raw = torch.load('dataLoader_1784578578190_speech_commands_mfcc20_test.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Dataset file not found: ' + 'dataLoader_1784578578190_speech_commands_mfcc20_test.pt' + '. '
        'This file does not exist yet — if it is a hand-built stimulus/dataset, generate it first (see the notebook guide\'s data-preparation step), then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt dataset ' + 'dataLoader_1784578578190_speech_commands_mfcc20_test.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels. '
        'Regenerate the dataset in one of those formats, then retry.'
    ) from exc
# Handle TensorDataset, dict, raw tuple, or a bare tensor (e.g. an
# unlabeled stimulus/spike-train tensor with no dataset wrapper)
_ds = _raw if hasattr(_raw, 'tensors') else TensorDataset(*_raw) if isinstance(_raw, (tuple, list)) else TensorDataset(_raw) if torch.is_tensor(_raw) else TensorDataset(_raw['data'], _raw['labels'])

train_loader = DataLoader(_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(_ds, batch_size=32, shuffle=False,          num_workers=0)
print(f'.pt dataset: {len(_ds)} samples, batch_size=32')

.pt dataset: 11005 samples, batch_size=32


## Architecture

Network compiled from CNL spec via NIR.

In [5]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named keyword_spotting.
# Network with 1 input, 7 hidden nodes, 1 output.
# flow: mfcc → encode → w_encode_hidden → hidden → w_hidden_association → association → w_association_readout → readout → keyword

# Layers:
Define an input port named mfcc with shape (20,).
Define a LIF neuron named encode with time constant shape (20,), resistance shape (20,), leak voltage shape (20,), and firing threshold shape (20,).
Define a linear transformation named w_encode_hidden with weight matrix shape (256, 20).
Define a LIF neuron named hidden with time constant shape (256,), resistance shape (256,), leak voltage shape (256,), and firing threshold shape (256,).
Define a linear transformation named w_hidden_association with weight matrix shape (256, 256).
Define a LIF neuron named association with time constant shape (256,), resistance shape (256,), leak voltage shape (256,), and firing threshold shape (256,).
Define a linear transformation named w_association_readout with weight matrix shape (35, 256).
Define a LIF neuron named readout with time constant shape (35,), resistance shape (35,), leak voltage shape (35,), and firing threshold shape (35,).
Define an output port named keyword with shape (35,).

# Connections:
mfcc connects to encode.
encode connects to w_encode_hidden.
w_encode_hidden connects to hidden.
hidden connects to w_hidden_association.
w_hidden_association connects to association.
association connects to w_association_readout.
w_association_readout connects to readout.
readout connects to keyword.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

Network: 9 nodes, 8 edges


In [6]:
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

from snntorch import surrogate
spike_grad = surrogate.fast_sigmoid(slope=25.0)

_w = np.load('weights_snntorch_sim_122e3c99.npz')  # weights file saved alongside this notebook

_expected_weight_keys = ['w_encode_hidden_weight', 'w_hidden_association_weight', 'w_association_readout_weight']
_missing_weight_keys = [k for k in _expected_weight_keys if k not in _w.files]
if _missing_weight_keys:
    raise RuntimeError(
        f"'weights_snntorch_sim_122e3c99.npz' is missing {_missing_weight_keys} — this weights "
        "file does not match the current network. Regenerate the notebook from "
        "the Architecture tab so its weights file matches this architecture."
    )

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # LIF population: 'encode'  (20 neurons)
        self.encode = snn.Leaky(beta=0.950000, threshold=1.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)
        # Linear layer: 'w_encode_hidden'  shape (256, 20)
        self.w_encode_hidden = nn.Linear(20, 256)
        # weight is all-zeros in NIR graph; keeping PyTorch default init
        self.w_encode_hidden.bias = None
        # LIF population: 'hidden'  (256 neurons)
        self.hidden = snn.Leaky(beta=0.950000, threshold=1.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)
        # Linear layer: 'w_hidden_association'  shape (256, 256)
        self.w_hidden_association = nn.Linear(256, 256)
        # weight is all-zeros in NIR graph; keeping PyTorch default init
        self.w_hidden_association.bias = None
        # LIF population: 'association'  (256 neurons)
        self.association = snn.Leaky(beta=0.950000, threshold=1.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)
        # Linear layer: 'w_association_readout'  shape (35, 256)
        self.w_association_readout = nn.Linear(256, 35)
        # weight is all-zeros in NIR graph; keeping PyTorch default init
        self.w_association_readout.bias = None
        # LIF population: 'readout'  (35 neurons)
        self.readout = snn.Leaky(beta=0.950000, threshold=1.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)

    def forward(self, x):
        # initialise hidden states
        self.encode.init_leaky()
        self.hidden.init_leaky()
        self.association.init_leaky()
        self.readout.init_leaky()
        # x: (T,B,C,H,W) time-first from tonic, or (B,C,H,W) for a single frame
        if x.dim() == 4:
            x = x.unsqueeze(0)  # (B,C,H,W) → (1,B,C,H,W)
        elif x.dim() == 3 and x.shape[0] <= x.shape[1]:
            x = x.swapaxes(0, 1)  # (B,T,N) → (T,B,N) synthetic spike batches
        elif x.dim() == 2:
            x = x.unsqueeze(0).expand(globals().get('num_steps', 1), -1, -1)  # (B,F) static features → (T,B,F), repeated every step (rate coding)
        _x_seq = x
        spk_rec = []
        mem_rec = []
        for t in range(_x_seq.shape[0]):
            x = _x_seq[t]
            spk_encode = self.encode(x)
            x = spk_encode
            x = self.w_encode_hidden(x)
            spk_hidden = self.hidden(x)
            x = spk_hidden
            x = self.w_hidden_association(x)
            spk_association = self.association(x)
            x = spk_association
            x = self.w_association_readout(x)
            spk_readout = self.readout(x)
            mem_rec.append(getattr(self.readout, 'mem', spk_readout).clone())
            x = spk_readout
            spk_rec.append(x)
        mem_out = torch.stack(mem_rec, dim=0) if mem_rec else x
        return torch.stack(spk_rec, dim=0), mem_out  # (T, batch, out), membrane trace or last activation


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

Net: 79616 parameters


## Train

In [7]:
num_steps = 25
optimizer = torch.optim.Adam(
    net.parameters(), lr=0.001, weight_decay=0.0,
    betas=(0.9, 0.999)
)
_vl_every_n_epochs = 1
_vl_save_best_checkpoint = True
_vl_checkpoint_metric = 'val_accuracy'
_vl_checkpoint_mode = 'max'
_vl_best = float('-inf')
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _val_raw = torch.load('dataLoader_1784576984130_speech_commands_mfcc20_train.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Validation dataset file not found: ' + 'dataLoader_1784576984130_speech_commands_mfcc20_train.pt' + '. '
        'Generate it first, then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt validation dataset ' + 'dataLoader_1784576984130_speech_commands_mfcc20_train.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels.'
    ) from exc
_val_ds = _val_raw if hasattr(_val_raw, 'tensors') else TensorDataset(*_val_raw) if isinstance(_val_raw, (tuple, list)) else TensorDataset(_val_raw) if torch.is_tensor(_val_raw) else TensorDataset(_val_raw['data'], _val_raw['labels'])

val_loader = DataLoader(_val_ds, batch_size=32, shuffle=False, num_workers=0)
print(f'.pt validation dataset: {len(_val_ds)} samples, batch_size=32')
net.train()
for epoch in range(50):
    _epoch_loss_sum = 0.0; _epoch_batches = 0
    for batch_idx, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        # Spike encoder: rate coding (input treated as spikes)
        spk_out, mem_out = net(data)
        loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.0)
        loss_val = loss_fn(spk_out.sum(0), targets)
        # spike output (spk_out) captured from forward pass
        loss_val.backward()
        optimizer.step()
        _epoch_loss_sum += loss_val.item(); _epoch_batches += 1
    avg_loss = _epoch_loss_sum / _epoch_batches if _epoch_batches else 0.0
    _nmtk_emit(
        epoch=epoch + 1,
        total=50,
        loss=avg_loss,
        accuracy=None,
        layer_rates={'output': float(spk_out.float().mean().item())},
        phase='train',
    )
    if (epoch + 1) % _vl_every_n_epochs == 0:
        net.eval()
        _vl_correct = 0
        _vl_total = 0
        _vl_loss_sum = 0.0
        _vl_batches = 0
        with torch.no_grad():
            for _vl_data, _vl_targets in val_loader:
                _vl_spk_out, _vl_mem_out = net(_vl_data)
                _vl_loss_sum += loss_fn(_vl_spk_out.sum(0), _vl_targets).item(); _vl_batches += 1
                _vl_correct += (_vl_spk_out.sum(0).argmax(1) == _vl_targets).sum().item()
                _vl_total += _vl_targets.size(0)
        val_loss = _vl_loss_sum / _vl_batches if _vl_batches else 0.0
        val_accuracy = _vl_correct / _vl_total if _vl_total else 0.0
        _nmtk_emit(
            epoch=epoch + 1,
            total=50,
            loss=(val_loss if val_loss is not None else 0.0),
            accuracy=val_accuracy,
            layer_rates={},
            phase='val',
        )
        _vl_metric_value = (
            val_accuracy if _vl_checkpoint_metric == 'val_accuracy'
            else (val_loss if val_loss is not None else val_accuracy)
        )
        _vl_improved = (
            _vl_metric_value >= _vl_best if _vl_checkpoint_mode == 'max'
            else _vl_metric_value <= _vl_best
        )
        if _vl_improved:
            _vl_best = _vl_metric_value
            if _vl_save_best_checkpoint:
                torch.save(net.state_dict(), 'best_model.pt')
        net.train()

.pt validation dataset: 94824 samples, batch_size=32
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 50, "loss": 3.4877508885638657, "accuracy": null, "layer_spike_rates": {"output": 0.005911330226808786}, "phase": "train"}
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 50, "loss": 3.402657081924791, "accuracy": 0.09519741837509492, "layer_spike_rates": {}, "phase": "val"}
{"__nmtk_progress__": true, "epoch": 2, "total_epochs": 50, "loss": 3.306643427804459, "accuracy": null, "layer_spike_rates": {"output": 0.011546798050403595}, "phase": "train"}
{"__nmtk_progress__": true, "epoch": 2, "total_epochs": 50, "loss": 3.3110569333016633, "accuracy": 0.10154602210410867, "layer_spike_rates": {}, "phase": "val"}
{"__nmtk_progress__": true, "epoch": 3, "total_epochs": 50, "loss": 3.197219944277475, "accuracy": null, "layer_spike_rates": {"output": 0.015408867038786411}, "phase": "train"}
{"__nmtk_progress__": true, "epoch": 3, "total_epochs": 50, "loss": 3.2769922592781495, "

## Evaluate

In [8]:
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _raw = torch.load('dataLoader_1784578578190_speech_commands_mfcc20_test.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Dataset file not found: ' + 'dataLoader_1784578578190_speech_commands_mfcc20_test.pt' + '. '
        'This file does not exist yet — if it is a hand-built stimulus/dataset, generate it first (see the notebook guide\'s data-preparation step), then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt dataset ' + 'dataLoader_1784578578190_speech_commands_mfcc20_test.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels. '
        'Regenerate the dataset in one of those formats, then retry.'
    ) from exc
# Handle TensorDataset, dict, raw tuple, or a bare tensor (e.g. an
# unlabeled stimulus/spike-train tensor with no dataset wrapper)
_ds = _raw if hasattr(_raw, 'tensors') else TensorDataset(*_raw) if isinstance(_raw, (tuple, list)) else TensorDataset(_raw) if torch.is_tensor(_raw) else TensorDataset(_raw['data'], _raw['labels'])

train_loader = DataLoader(_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(_ds, batch_size=32, shuffle=False,          num_workers=0)
print(f'.pt dataset: {len(_ds)} samples, batch_size=32')
try:
    net.load_state_dict(torch.load('best_model.pt', weights_only=True))
    print('Loaded best checkpoint from best_model.pt')
except FileNotFoundError:
    print("best_model.pt not found — evaluating with current in-memory weights "
          "(enable 'Save Best Checkpoint' on the Validation Loop node and train first).")
correct = 0; total = 0
net.eval()
with torch.no_grad():
    for batch_idx, (data, targets) in enumerate(test_loader):
        spk_out, mem_out = net(data)
        
        correct += (spk_out.sum(0).argmax(1) == targets).sum().item()
        total   += targets.size(0)
        _nmtk_emit(
            epoch=batch_idx + 1,
            total=len(test_loader),
            loss=0.0,
            accuracy=(correct / total if total else None),
            layer_rates={'output': float(spk_out.float().mean().item())},
            phase='eval',
        )
print(f'Accuracy (top-1): {correct/total:.2%}' if total else 'Accuracy (top-1): n/a')
_nmtk_emit(
    epoch=1,
    total=1,
    loss=0.0,
    accuracy=(correct / total if total else None),
    layer_rates={'output': float(spk_out.float().mean().item())},
    phase='eval',
)

.pt dataset: 11005 samples, batch_size=32
Loaded best checkpoint from best_model.pt
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 344, "loss": 0.0, "accuracy": 0.125, "layer_spike_rates": {"output": 0.017000000923871994}, "phase": "eval"}
{"__nmtk_progress__": true, "epoch": 2, "total_epochs": 344, "loss": 0.0, "accuracy": 0.09375, "layer_spike_rates": {"output": 0.018321428447961807}, "phase": "eval"}
{"__nmtk_progress__": true, "epoch": 3, "total_epochs": 344, "loss": 0.0, "accuracy": 0.08333333333333333, "layer_spike_rates": {"output": 0.013785714283585548}, "phase": "eval"}
{"__nmtk_progress__": true, "epoch": 4, "total_epochs": 344, "loss": 0.0, "accuracy": 0.078125, "layer_spike_rates": {"output": 0.01971428655087948}, "phase": "eval"}
{"__nmtk_progress__": true, "epoch": 5, "total_epochs": 344, "loss": 0.0, "accuracy": 0.075, "layer_spike_rates": {"output": 0.017142856493592262}, "phase": "eval"}
{"__nmtk_progress__": true, "epoch": 6, "total_epochs": 344, "loss": 0.0,

In [9]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')

This application is used to convert notebook files (*.ipynb) to various other
formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePreprocess

[NbConvertApp] WARNING | pattern '__file__' matched no files
